# The data model and dunder methods

Python’s data model is the layer that lets your own classes behave like built-in types. When you call `len(obj)`, compare two objects, use an object in a set, or iterate over it in a loop, Python is really looking for special methods such as `__len__`, `__eq__`, `__hash__`, and `__iter__`.

The power of dunder methods is that they make your abstractions feel natural. The danger is that incorrect implementations create subtle bugs, especially around hashing, equality, ordering, and context management.

As you move through these notebooks, think in terms of contracts: when Python calls a dunder method, what promise is your class making back to the rest of the language?

## Visual model

```text
your code -> len(x) / x == y / for item in x
             -> __len__ / __eq__ / __iter__
```

## How to use this notebook

Read the concept notes first, then run the code cells one at a time. After each run, change an input, prediction, or line of code and rerun it. Intermediate Python becomes easier when you treat every notebook as a place to test a mental model, not just a place to read finished answers.

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.


---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 2. `__eq__` and `__hash__` are a pair

In [ ]:
class Point:
    def __init__(self, x: float, y: float) -> None:
        self.x, self.y = x, y

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Point):
            return NotImplemented          # let the OTHER side try
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self) -> int:
        return hash((self.x, self.y))      # hash what __eq__ compares

**Three rules, and violating any one causes silent, hard-to-find bugs:**

1. **Equal objects must have equal hashes.** If not, a dict looks in the wrong
   bucket and your key becomes unreachable — present in memory, invisible to
   lookup.
2. **Only hash immutable state.** If a hashed attribute changes after insertion,
   the same unreachable-entry bug appears.
3. **Defining `__eq__` sets `__hash__` to `None`.** Python does this
   deliberately, because a default identity hash would violate rule 1. Your
   class becomes unhashable unless you define `__hash__` too:

In [ ]:
class Bad:
    def __eq__(self, other): return True

{Bad()}     # TypeError: unhashable type: 'Bad'

**Return `NotImplemented`, not `False`, for unknown types.** `NotImplemented`
tells Python "I do not know", so it tries `other.__eq__(self)` before falling
back to identity. Returning `False` claims authority you do not have and breaks
comparison with types written to interoperate with yours.

`@dataclass` generates all of this correctly (Module 11), which is the main
reason to use it.

---

## Concept 4. The container protocols

In [ ]:
class Deck:
    def __init__(self, cards: list[str]) -> None:
        self._cards = list(cards)

    def __len__(self) -> int:                 # len(deck)
        return len(self._cards)

    def __getitem__(self, index):             # deck[0], deck[1:3]
        return self._cards[index]             # slices work for free

    def __setitem__(self, index, value) -> None:
        self._cards[index] = value

    def __delitem__(self, index) -> None:
        del self._cards[index]

    def __contains__(self, card: str) -> bool:  # "AS" in deck
        return card in self._cards

    def __iter__(self):                        # for card in deck
        return iter(self._cards)

    def __reversed__(self):                    # reversed(deck)
        return reversed(self._cards)

**`__getitem__` alone gives you a great deal.** Without `__iter__`, Python falls
back to calling `__getitem__` with 0, 1, 2, ... until `IndexError`. So iteration,
`in`, `list()`, and unpacking all work from `__getitem__` alone. This is the old
protocol, kept for compatibility — implement `__iter__` explicitly anyway,
because the fallback only works for integer-indexed sequences and produces
confusing errors when it does not apply.

**Handling slices:** `__getitem__` receives a `slice` object for `deck[1:3]`.
Delegating to a list (as above) handles it automatically. If you build the result
yourself, return **your own type** for a slice and a single element for an int:

In [ ]:
def __getitem__(self, index):
    if isinstance(index, slice):
        return type(self)(self._cards[index])     # type(self), not Deck --
    return self._cards[index]                     # subclasses get their type

---

## Concept 7. Operators

In [ ]:
class Vector:
    def __init__(self, x: float, y: float) -> None:
        self.x, self.y = x, y

    def __add__(self, other: "Vector") -> "Vector":
        if not isinstance(other, Vector):
            return NotImplemented
        return Vector(self.x + other.x, self.y + other.y)

    def __mul__(self, scalar: float) -> "Vector":
        if not isinstance(scalar, (int, float)):
            return NotImplemented
        return Vector(self.x * scalar, self.y * scalar)

    __rmul__ = __mul__            # makes 3 * v work as well as v * 3

    def __neg__(self) -> "Vector":
        return Vector(-self.x, -self.y)

    def __abs__(self) -> float:
        return (self.x**2 + self.y**2) ** 0.5

**How Python resolves `a + b`:**

1. Try `type(a).__add__(a, b)`. If it returns `NotImplemented`, continue.
2. Try `type(b).__radd__(b, a)`. If that also returns `NotImplemented`:
3. `TypeError: unsupported operand type(s)`.

(With one refinement: if `type(b)` is a *subclass* of `type(a)`, the reflected
method is tried first, so a subclass can override its parent's behaviour.)

This is why `NotImplemented` matters. Returning it is how you say "not my
problem" and let the other operand try. Note the trap: `NotImplemented` is
**truthy**, so accidentally returning it from `__eq__` and using the result in an
`if` gives you a silent wrong answer plus a `DeprecationWarning`.

**In-place operators** (`__iadd__` etc.) should mutate and `return self` — for a
mutable type. For an immutable one, omit them and Python falls back to
`__add__` plus rebinding. This is exactly Module 02's list-versus-tuple `+=`
distinction, now from the implementer's side.

Only overload operators where the meaning is obvious. `Vector + Vector` is
clear. `User + User` is not, and a `merge()` method would be better.

---

## Concept 9. The whole map

| Group | Methods |
|---|---|
| Representation | `__repr__` `__str__` `__format__` `__bytes__` |
| Comparison | `__eq__` `__ne__` `__lt__` `__le__` `__gt__` `__ge__` `__hash__` |
| Container | `__len__` `__getitem__` `__setitem__` `__delitem__` `__contains__` `__reversed__` |
| Iteration | `__iter__` `__next__` `__aiter__` `__anext__` |
| Numeric | `__add__` `__sub__` `__mul__` `__truediv__` `__floordiv__` `__mod__` `__pow__` `__neg__` `__abs__` `__round__` and the `__r*__` / `__i*__` variants |
| Conversion | `__bool__` `__int__` `__float__` `__index__` `__complex__` |
| Context | `__enter__` `__exit__` `__aenter__` `__aexit__` |
| Callable | `__call__` |
| Attributes | `__getattr__` `__getattribute__` `__setattr__` `__delattr__` `__dir__` |
| Descriptors | `__get__` `__set__` `__delete__` `__set_name__` |
| Class machinery | `__init__` `__new__` `__init_subclass__` `__class_getitem__` `__slots__` |
| Copying | `__copy__` `__deepcopy__` `__reduce__` |
| Pattern matching | `__match_args__` |

You do not need to memorise this. You need to know it exists, so that when you
want your type to work with some piece of syntax, you look up which method
provides it.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: `__repr__` and `__str__`
- Section 2: `__eq__` and `__hash__` are a pair
- Section 3: Ordering
- Section 4: The container protocols
- Section 5: Iteration
- Section 6: Context managers
- Section 7: Operators
- Section 8: `__call__`, `__bool__`, `__format__`
- Section 9: The whole map

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

from collections.abc import Hashable, Iterator, Mapping
from typing import Any

---

## `FrozenDict`

An immutable, hashable mapping.

In [ ]:
class FrozenDict(Mapping[str, Any]):
    """An immutable, hashable mapping.

    Inheriting from collections.abc.Mapping gives you get, keys, values, items,
    __contains__, __eq__ and __ne__ for free, provided you implement the three
    abstract methods. Look up which three before starting -- finding that out is
    part of the exercise.

    TODO 1  __init__ accepting the same argument forms dict does:
              FrozenDict()
              FrozenDict({"a": 1})
              FrozenDict([("a", 1)])
              FrozenDict(a=1, b=2)
            Store a private copy. A caller who passes a dict and then mutates it
            must not change your FrozenDict (Module 08).

    TODO 2  The three abstract methods Mapping requires.

    TODO 3  __hash__. This is the hard part and the point of the exercise.
            - dicts have no defined order for hashing purposes, so
              {"a": 1, "b": 2} and {"b": 2, "a": 1} MUST hash equally
            - the hash must be consistent with the __eq__ that Mapping gave you
            - it must raise TypeError if any VALUE is unhashable, because
              FrozenDict({"a": [1,2]}) cannot honestly claim to be hashable
            - cache it: hashing is O(n) and a hashable object gets hashed a lot
            Hint: frozenset(self.items()) solves the ordering problem in one
            step. Work out why before using it.

    TODO 4  __repr__ that round-trips.

    TODO 5  Explicitly BLOCK mutation with clear errors:
            __setitem__, __delitem__, and any attribute assignment after
            construction. A silent "it just does not have that method"
            AttributeError is much less helpful than a message saying the type
            is immutable.

    TODO 6  Two derivation methods that return NEW FrozenDicts:
              with_(key, value)      -> a copy plus one entry
              without(key)           -> a copy minus one entry
            Immutable types need these, or they are unusable.

    TODO 7  Then answer:
            - MappingProxyType also gives a read-only mapping. Name two things
              FrozenDict does that MappingProxyType does not.
            - what should FrozenDict({"a": 1}) == {"a": 1} return? Justify.
    """

---

## `verify`

_verify_

In [ ]:
def verify() -> None:
    fd = FrozenDict(a=1, b=2)

    assert fd["a"] == 1
    assert fd.get("z", "default") == "default"
    assert len(fd) == 2
    assert set(fd) == {"a", "b"}
    assert set(fd.keys()) == {"a", "b"}
    assert sorted(fd.values()) == [1, 2]
    assert ("a", 1) in set(fd.items())
    assert "a" in fd and "z" not in fd

    assert FrozenDict({"a": 1}) == FrozenDict(a=1)
    assert FrozenDict(a=1, b=2) == FrozenDict(b=2, a=1)
    assert hash(FrozenDict(a=1, b=2)) == hash(FrozenDict(b=2, a=1)), (
        "insertion order must not affect the hash"
    )

    d = {FrozenDict(a=1): "value"}
    assert d[FrozenDict(a=1)] == "value"
    assert len({FrozenDict(a=1), FrozenDict(a=1)}) == 1

    for op, args in [("__setitem__", ("a", 9)), ("__delitem__", ("a",))]:
        try:
            getattr(fd, op)(*args)
        except TypeError as exc:
            assert "immutable" in str(exc).lower(), str(exc)
        else:
            raise AssertionError(f"{op} should have raised")

    try:
        fd.new_attribute = 1        # type: ignore[attr-defined]
    except (AttributeError, TypeError):
        pass
    else:
        raise AssertionError("attribute assignment should be blocked")

    source = {"a": 1}
    fd2 = FrozenDict(source)
    source["b"] = 2
    assert "b" not in fd2, "constructor must copy its input"

    assert fd.with_("c", 3) == FrozenDict(a=1, b=2, c=3)
    assert fd.without("a") == FrozenDict(b=2)
    assert fd == FrozenDict(a=1, b=2), "derivation must not mutate the original"

    try:
        hash(FrozenDict(bad=[1, 2]))
    except TypeError:
        pass
    else:
        raise AssertionError("a FrozenDict with unhashable values must not hash")

    assert eval(repr(fd)) == fd    # noqa: S307

    print("all FrozenDict checks passed")

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    verify()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.